In [1]:
import os
import pandas as pd
import glob
import csv
import cv2

# Dataset Combination

In [2]:
input_dir = r"D:\Downloads\Computer_Vision\Capstone_Project\data"
output_dir = r"D:\Downloads\Computer_Vision\Capstone_Project\data\combined_dataset"

IMAGE_DIR = os.path.join(output_dir, "images")
MASK_DIR = os.path.join(output_dir, "masks")
CSV_PATH = os.path.join(output_dir, "data_division.csv")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

In [3]:
from tqdm import tqdm

In [5]:
def process_massachusetts(writer, output_size=(1024, 1024)):

    dataset_root = os.path.join(input_dir, 'Massachusetts_Roads_Dataset', 'tiff')
    split_names = ['train', 'val', 'test']
    file_index = 0

    for split_name in split_names:
        image_folder = os.path.join(dataset_root, split_name)
        label_folder = os.path.join(dataset_root, f"{split_name}_labels")

        image_files = sorted(glob.glob(os.path.join(image_folder, "*.tiff")))

        for image_file in tqdm(image_files):
            filename = os.path.splitext(os.path.basename(image_file))[0]
            label_file = os.path.join(label_folder, f"{filename}.tif")

            if not os.path.exists(label_file):
                print(f"Missing label file: {label_file}")
                continue

            image_data = cv2.imread(image_file, cv2.IMREAD_COLOR)
            label_data = cv2.imread(label_file, cv2.IMREAD_GRAYSCALE)

            if image_data is None or label_data is None:
                print(f"Unable to load image or label:\n  Image: {image_file}\n  Label: {label_file}")
                continue

            resized_image = cv2.resize(
                image_data,
                output_size,
                interpolation=cv2.INTER_LINEAR
            )

            resized_label = cv2.resize(
                label_data,
                output_size,
                interpolation=cv2.INTER_NEAREST
            )

            output_image = f"mass_{file_index:05d}.jpg"
            output_label = f"mass_{file_index:05d}.png"

            image_output_path = os.path.join(IMAGE_DIR, output_image)
            label_output_path = os.path.join(MASK_DIR, output_label)

            cv2.imwrite(image_output_path, resized_image)
            cv2.imwrite(label_output_path, resized_label)

            writer.writerow({
                "filename": output_image,
                "maskname": output_label,
                "split": split_name
            })

            file_index += 1

    print("Massachusetts dataset preprocessing completed successfully.")

In [6]:
def process_deepglobe(writer, output_size=(1024, 1024)):
    dataset_folder = os.path.join(input_dir, 'DeepGlobe_Road_Extraction_Dataset', 'train')

    satellite_images = sorted(
        glob.glob(os.path.join(dataset_folder, '*_sat.jpg'))
    )

    for image_id, image_file in enumerate(tqdm(satellite_images)):
        label_file = image_file.replace('_sat.jpg', '_mask.png')

        if not os.path.exists(label_file):
            print(f"Skipped sample because the mask file is missing: {label_file}")
            continue

        image_data = cv2.imread(image_file, cv2.IMREAD_COLOR)
        label_data = cv2.imread(label_file, cv2.IMREAD_GRAYSCALE)

        if image_data is None or label_data is None:
            print(
                f"Failed to read image/mask pair:\n"
                f"  Image: {image_file}\n"
                f"  Mask : {label_file}"
            )
            continue

        resized_image = cv2.resize(
            image_data,
            output_size,
            interpolation=cv2.INTER_LINEAR
        )

        resized_label = cv2.resize(
            label_data,
            output_size,
            interpolation=cv2.INTER_NEAREST
        )

        image_name = f"deep_{image_id:05d}.jpg"
        label_name = f"deep_{image_id:05d}.png"

        image_save_path = os.path.join(IMAGE_DIR, image_name)
        label_save_path = os.path.join(MASK_DIR, label_name)

        cv2.imwrite(image_save_path, resized_image)
        cv2.imwrite(label_save_path, resized_label)

        writer.writerow({
            "filename": image_name,
            "maskname": label_name,
            "split": "train"
        })

    print("DeepGlobe dataset conversion completed successfully.")

In [7]:
with open(CSV_PATH, mode='w', newline='') as csvfile:
    fieldnames = ['filename', 'maskname', 'split']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    process_massachusetts(writer)
    process_deepglobe(writer)
    
print(f"\nDone!")

100%|██████████| 49/49 [00:02<00:00, 18.25it/s]


Massachusetts dataset preprocessing completed successfully.


100%|██████████| 6226/6226 [03:57<00:00, 26.20it/s]

DeepGlobe dataset conversion completed successfully.

Done!


# Data Division

In [8]:
df = pd.read_csv(CSV_PATH)

df = df.sample(frac=1, random_state=21).reset_index(drop=True)

n_total = len(df)
n_train = int(n_total * 0.7)
n_val = int(n_total * 0.15)

df.loc[:n_train-1, 'split'] = 'train'
df.loc[n_train:n_train+n_val-1, 'split'] = 'val'
df.loc[n_train+n_val:, 'split'] = 'test'

df.to_csv(CSV_PATH, index=False)

print(f"Division: {df['split'].value_counts().to_dict()}")

Division: {'train': 5177, 'test': 1111, 'val': 1109}
